**PASS: fresh instructor execution.** All saved outputs below came from this notebook's actual code. Run UTC: `2026-11-04T15:25:36.558567+00:00`; executed 7/7 code cells; 7 output objects. This validates the displayed computations; the described synthetic scope still applies.
<!-- canonical-execution-status:end -->

# 02 · Training safeguards and actual sequence generation

**Instructor worked notebook · select 8–10 minutes for the interview; use the rest in a follow-up lab.**
Question: how do we distinguish a useful training objective, correct information flow, and a search decision?
Predictions, assertions, raw outputs, and interpretation are kept together. All controls are synthetic and completely declared.

In [1]:
from pathlib import Path
import sys, math, time, statistics, json
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
assert (ROOT / 'src/vgu_transformers.py').is_file(), 'Run from REHEARSAL_READY or its notebooks directory.'
sys.path.insert(0, str(ROOT / 'src'))
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import display
import vgu_transformers as vgu
from vgu_transformers import *
torch.manual_seed(9)
torch.set_num_threads(1)
FIGURES=ROOT/'verification/figures'
FIGURES.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,
                     'axes.spines.top':False,'axes.spines.right':False,
                     'figure.facecolor':'white','axes.facecolor':'white'})
print('PyTorch:',torch.__version__,'| device: CPU | threads:',torch.get_num_threads())
print('Actual imported source:',Path(vgu.__file__).resolve().relative_to(ROOT))
assert Path(vgu.__file__).resolve()==(ROOT/'src/vgu_transformers.py').resolve()

PyTorch: 2.10.0+cpu | device: CPU | threads: 1
Actual imported source: src/vgu_transformers.py


## 1. Test causality by changing the future

A triangular picture alone cannot prove the implementation uses the mask correctly.
Change keys and values at positions 3–4 by a large amount and verify that outputs at positions 0–2 do not change.
The unmasked control should change; otherwise the experiment would not diagnose the mask.

In [2]:
q=torch.randn(1,2,5,4); k=torch.randn(1,2,5,4); values=torch.randn(1,2,5,6)
altered_k=k.clone();altered_k[:,:,3:]=100.
altered_values=values.clone();altered_values[:,:,3:]=-100.
mask=causal_additive_mask(5)
original,w=scaled_dot_product_manual(q,k,values,mask)
changed,_=scaled_dot_product_manual(q,altered_k,altered_values,mask)
unmasked,_=scaled_dot_product_manual(q,k,values)
unmasked_changed,_=scaled_dot_product_manual(q,altered_k,altered_values)
causal_diff=float((original[:,:,:3]-changed[:,:,:3]).abs().max())
control_diff=float((unmasked[:,:,:3]-unmasked_changed[:,:,:3]).abs().max())
print('earlier-output max difference with causal mask:',causal_diff)
print('earlier-output max difference without mask:',control_diff)
torch.testing.assert_close(original[:,:,:3],changed[:,:,:3])
assert control_diff>1 and w[...,torch.triu(torch.ones(5,5,dtype=torch.bool),1)].abs().sum()==0

earlier-output max difference with causal mask: 0.0
earlier-output max difference without mask: 99.96993255615234


## 2. Padding keys, fully blocked queries, and gradients

A padding **key** mask prevents invalid source positions from contributing to attention.
Here our explicit all-masked policy is zero weights and zero output; it avoids NaNs but is an implementation choice.
Valid rows sum to one. Fully blocked rows sum to zero and should not be treated as valid predictions.
Additive masks are used throughout this teaching API; boolean conventions vary between PyTorch attention APIs.

In [3]:
q=torch.randn(1,2,3,4,requires_grad=True)
k=torch.randn(1,2,5,4); values=torch.randn(1,2,5,6)
padding_mask=torch.tensor([0.,0.,0.,-math.inf,-math.inf]).view(1,1,1,5)
original,_=scaled_dot_product_manual(q,k,values,padding_mask)
changed_k=k.clone();changed_k[:,:,3:]=99.
changed_v=values.clone();changed_v[:,:,3:]=-99.
changed,_=scaled_dot_product_manual(q,changed_k,changed_v,padding_mask)
torch.testing.assert_close(original,changed)
print('padding-key perturbation max difference:',float((original-changed).detach().abs().max()))
all_blocked=torch.full((3,5),-math.inf)
zero,zero_weights=scaled_dot_product_manual(q,k,values,all_blocked)
zero.sum().backward()
print('all-blocked weights sum:',float(zero_weights.detach().sum()),
      'output sum:',float(zero.detach().sum()),'query gradient norm:',float(q.grad.norm()))
assert torch.isfinite(zero).all() and torch.isfinite(q.grad).all()
assert zero.abs().sum()==0 and zero_weights.abs().sum()==0 and q.grad.abs().sum()==0

padding-key perturbation max difference: 0.0
all-blocked weights sum: 0.0 output sum: 0.0 query gradient norm: 0.0


## 3. Loss padding is a separate operation

`ignore_index=PAD` removes padding labels from the mean cross-entropy and gives those logits zero loss gradient.
It does not add an attention mask. Our all-ignored-label policy is a differentiable zero loss.
In a real training loop, report supervised token counts and usually avoid optimizing an empty batch.

In [4]:
logits=torch.randn(1,3,7,requires_grad=True)
labels=torch.tensor([[0,PAD,2]])
loss=masked_lm_loss(logits,labels,ignore_index=PAD)
independent=F.cross_entropy(logits[:,[0,2]].reshape(-1,7),torch.tensor([0,2]))
torch.testing.assert_close(loss,independent)
loss.backward()
print('supervised tokens:',int(labels.ne(PAD).sum()),'loss:',float(loss.detach()))
print('ignored-position loss gradient:',logits.grad[:,1].tolist())
assert logits.grad[:,1].abs().sum()==0
empty_logits=torch.randn(1,2,7,requires_grad=True)
empty=masked_lm_loss(empty_logits,torch.full((1,2),PAD),ignore_index=PAD)
empty.backward()
print('all-ignored loss:',float(empty.detach()),'total gradient:',float(empty_logits.grad.abs().sum()))
assert empty.item()==0 and empty.requires_grad and empty_logits.grad.abs().sum()==0

supervised tokens: 2 loss: 3.155644416809082
ignored-position loss gradient: [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]
all-ignored loss: 0.0 total gradient: 0.0


## 4. Actually run greedy and beam search on a counterexample

Fix a **two-token horizon**, with no EOS in this control. The first distribution is $P(A)=0.6$, $P(B)=0.4$.
After A, $P(X|A)=P(Y|A)=0.5$; after B, $P(X|B)=1$.
The complete sequence probabilities are AX=0.30, AY=0.30, BX=0.40.

We feed these distributions through the same generation functions used by the image-to-sequence demo.
Beam width is **3**, length exponent **alpha=0**, so ranking uses the raw sum of log probabilities.
Ties under A may choose X or Y; both have probability 0.30. The current PyTorch argmax control selects X.
**Ask:** why does the locally best first token lose here? Does a higher model score establish real-world correctness?

In [5]:
from teaching_demos import FixedHorizonTree, EOSControl, sequence_log_probability
tree=FixedHorizonTree(); dummy=torch.zeros(1,1,1,5,5)
greedy=greedy_generate(tree,dummy,max_len=2)[0].tolist()
beam,beam_logscore=beam_generate_one(tree,dummy,beam_width=3,alpha=0.,max_len=2)
names={0:'A',1:'B',2:'X',3:'Y',BOS:'BOS',EOS:'EOS',PAD:'PAD'}
greedy_probability=math.exp(sequence_log_probability(tree,dummy,greedy))
beam_probability=math.exp(beam_logscore)
print('greedy tokens:',[names[t] for t in greedy],'joint probability:',greedy_probability)
print('beam width 3, alpha 0:',[names[t] for t in beam],'joint probability:',beam_probability)
assert greedy==[BOS,0,2] and beam==[BOS,1,2]
assert math.isclose(greedy_probability,.3,abs_tol=1e-6)
assert math.isclose(beam_probability,.4,abs_tol=1e-6)
assert math.isclose(beam_logscore,sequence_log_probability(tree,dummy,beam),abs_tol=1e-6)
print('The observed result disproves universal greedy sequence optimality; it does not prove beam always finds a global optimum.')

greedy tokens: ['BOS', 'A', 'X'] joint probability: 0.30000001423898903
beam width 3, alpha 0: ['BOS', 'B', 'X'] joint probability: 0.40000001257540685
The observed result disproves universal greedy sequence optimality; it does not prove beam always finds a global optimum.


## 5. EOS is a separate stopping rule

The fixed-horizon control above intentionally excludes EOS. Now test two rows that finish at different times.
Row 0 emits EOS immediately; row 1 emits `1, 1, EOS`. The finished row is padded while the other continues.
Metrics remove the leading BOS and stop at the first EOS. Internal BOS or pre-EOS PAD remains a prediction error.

In [6]:
result=greedy_generate(EOSControl(),torch.zeros(2,3,1,5,5))
print('raw token rows:',result.tolist())
print('metric token rows:',[tokens_before_eos(row) for row in result.tolist()])
assert result.tolist()==[[BOS,EOS,PAD,PAD],[BOS,1,1,EOS]]
assert tokens_before_eos([BOS,0,BOS,PAD,EOS])==[0,BOS,PAD]
print('Internal special-token errors remain visible:',tokens_before_eos([BOS,0,BOS,PAD,EOS]))

raw token rows: [[4, 5, 6, 6], [4, 1, 1, 5]]
metric token rows: [[], [1, 1]]
Internal special-token errors remain visible: [0, 4, 6]


## 6. Optional: temperature sampling from the same logits

For each temperature, use the same base logits $\log[0.55,0.30,0.15]$ and compute softmax(logits/temperature).
Lower temperature concentrates this distribution; it does not add evidence that the modal token is correct.
Sample counts vary with the generator. A seed makes this demonstration repeatable in this environment.

In [7]:
base_logits=torch.log(torch.tensor([.55,.30,.15]))
for temperature in (.5,1.,2.):
    probabilities=torch.softmax(base_logits/temperature,-1)
    draws=torch.multinomial(probabilities,300,replacement=True,generator=torch.Generator().manual_seed(11))
    counts=torch.bincount(draws,minlength=3)
    print('temperature:',temperature,'probabilities:',[round(x,4) for x in probabilities.tolist()],
          '300 sample counts:',counts.tolist())
    assert counts.sum()==300 and probabilities.argmax()==0

temperature: 0.5 probabilities: [0.7289, 0.2169, 0.0542] 300 sample counts: [206, 80, 14]
temperature: 1.0 probabilities: [0.55, 0.3, 0.15] 300 sample counts: [146, 106, 48]
temperature: 2.0 probabilities: [0.4423, 0.3267, 0.231] 300 sample counts: [121, 100, 79]


## Exit ticket → a useful next project

Explain why a padding loss mask does not protect attention, why a causal invariance test is stronger than a mask picture,
and what objective beam search actually tries to improve.
Implement the separate student API, make the invariance tests pass, then evaluate greedy/beam on the same held-out examples with an explicit stopping and length-normalization policy.
Report both model score and task quality; diagnose their disagreements instead of assuming they coincide.